In [1]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta, timezone


# DEFINE COINS & INTERVAL

In [2]:
coins = ["BTCUSDT", "ETHUSDT", "BNBUSDT"]
interval = "15m"

print("Coins:", coins)
print("Interval:", interval)


Coins: ['BTCUSDT', 'ETHUSDT', 'BNBUSDT']
Interval: 15m


# DEFINE 5-YEAR TIME RANGE

In [3]:
# Current UTC time
end_time = datetime.now(timezone.utc)

# 5 years back
start_time = end_time - timedelta(days=5*365)

# Convert to milliseconds (Binance requirement)
start_timestamp = int(start_time.timestamp() * 1000)
end_timestamp = int(end_time.timestamp() * 1000)

print("Start:", start_time)
print("End:", end_time)


Start: 2021-02-25 04:57:27.553887+00:00
End: 2026-02-24 04:57:27.553887+00:00


# DEFINE BINANCE FETCH FUNCTION

In [4]:
BASE_URL = "https://api.binance.com/api/v3/klines"

def fetch_binance_klines(symbol, interval, start_ts, end_ts):
    
    all_data = []
    current_start = start_ts
    
    while current_start < end_ts:
        
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": current_start,
            "endTime": end_ts,
            "limit": 1000
        }
        
        response = requests.get(BASE_URL, params=params)
        data = response.json()
        
        if not data:
            break
        
        all_data.extend(data)
        
        last_open_time = data[-1][0]
        current_start = last_open_time + 1
        
        time.sleep(0.2)  # Avoid rate limit
    
    return all_data


# FETCH DATA FOR ALL COINS

In [6]:
all_coins_data = []

for coin in coins:
    
    print(f"\nFetching: {coin}")
    
    coin_data = fetch_binance_klines(
        symbol=coin,
        interval=interval,
        start_ts=start_timestamp,
        end_ts=end_timestamp
    )
    
    for row in coin_data:
        row.append(coin)
    
    all_coins_data.extend(coin_data)
    
    print(f"{coin} Done  | Candles: {len(coin_data)}")



Fetching: BTCUSDT
BTCUSDT Done  | Candles: 175135

Fetching: ETHUSDT
ETHUSDT Done  | Candles: 175135

Fetching: BNBUSDT
BNBUSDT Done  | Candles: 175135


# CONVERT TO DATAFRAME

In [7]:
df = pd.DataFrame(all_coins_data)

df.columns = [
    "OpenTime", "Open", "High", "Low", "Close", "Volume",
    "CloseTime", "QuoteAssetVolume", "NumberOfTrades",
    "TakerBuyBase", "TakerBuyQuote", "Ignore", "Symbol"
]

df = df[["OpenTime", "Open", "High", "Low", "Close", "Volume", "Symbol"]]

df["OpenTime"] = pd.to_datetime(df["OpenTime"], unit="ms")

df[["Open","High","Low","Close","Volume"]] = \
df[["Open","High","Low","Close","Volume"]].astype(float)

df.head()


,OpenTime,Open,High,Low,Close,Volume,Symbol
0,2021-02-25 05:00:00,49626.90,49838.20,49533.44,49763.70,705.610526,BTCUSDT
1,2021-02-25 05:15:00,49763.70,49923.19,49613.29,49701.93,388.999845,BTCUSDT
2,2021-02-25 05:30:00,49701.94,50264.73,49564.02,50002.21,676.267568,BTCUSDT
3,2021-02-25 05:45:00,50002.21,50449.07,49873.46,50388.60,718.398206,BTCUSDT
4,2021-02-25 06:00:00,50391.10,50540.97,50246.89,50422.20,551.371592,BTCUSDT


In [14]:
print("Shape:", df.shape)
print("Null values:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())


Shape: (525405, 7)
Null values:
 OpenTime    0
Open        0
High        0
Low         0
Close       0
Volume      0
Symbol      0
dtype: int64
Duplicates: 0


# SAVE AS FEATHER

In [15]:
df.to_feather("crypto_5years_15m.feather")


In [16]:
for coin in df["Symbol"].unique():
    
    df_coin = df[df["Symbol"] == coin].copy()
    
    file_name = f"{coin}_5years_15m.feather"
    
    df_coin.to_feather(file_name)
    
    print(f"Saved: {file_name} | Rows: {len(df_coin)}")


Saved: BTCUSDT_5years_15m.feather | Rows: 175135
Saved: ETHUSDT_5years_15m.feather | Rows: 175135
Saved: BNBUSDT_5years_15m.feather | Rows: 175135


 # BUILD MULTI-COIN MATRIX

In [18]:
# First sort properly
df = df.sort_values(["Symbol", "OpenTime"])

# Pivot to matrix format
matrix = df.pivot(index="OpenTime", columns="Symbol")

matrix.head()


Open                         High                     \
Symbol                BNBUSDT   BTCUSDT  ETHUSDT   BNBUSDT   BTCUSDT  ETHUSDT   
OpenTime                                                                        
2021-02-24 06:00:00  243.5559  50247.61  1627.42  248.3040  50719.78  1648.99   
2021-02-24 06:15:00  240.3751  49736.62  1617.69  242.5252  50147.00  1634.72   
2021-02-24 06:30:00  235.9794  49666.92  1612.40  241.8185  50076.92  1628.00   
2021-02-24 06:45:00  241.4877  50055.59  1627.31  244.8000  50427.00  1643.01   
2021-02-24 07:00:00  242.9551  50109.20  1637.58  245.0140  50181.33  1643.21   

                          Low                        Close                     \
Symbol                BNBUSDT   BTCUSDT  ETHUSDT   BNBUSDT   BTCUSDT  ETHUSDT   
OpenTime                                                                        
2021-02-24 06:00:00  239.7777  49593.46  1616.18  240.2982  49736.62  1618.05   
2021-02-24 06:15:00  235.5344  49456.00  1610.05  235.8546  49666.93  1612.79   
2021-02-24 06:30:00  235.8039  49573.42  1610.00  241.3975  50055.59  1627.75   
2021-02-24 06:45:00  240.5307  49927.57  1624.74  242.9589  50109.21  1637.60   
2021-02-24 07:00:00  240.2200  49548.47  1614.36  240.2269  49552.56  1614.58   

                         Volume                            
Symbol                  BNBUSDT      BTCUSDT      ETHUSDT  
OpenTime                                                   
2021-02-24 06:00:00  151180.501  1557.357006  13346.38967  
2021-02-24 06:15:00  182855.252  1309.230220  14505.60860  
2021-02-24 06:30:00   94903.977   926.348762  10067.88222  
2021-02-24 06:45:00  113713.304   873.182482  12905.22595  
2021-02-24 07:00:00   94152.152   779.167489  17812.04168

In [19]:
print("Matrix Shape:", matrix.shape)
print("Missing values:", matrix.isnull().sum().sum())


Matrix Shape: (175135, 15)
Missing values: 0


# LOAD & ALIGN ALL COINS

In [20]:
# Load all saved feather files
dfs = []

for coin in coins:
    
    file_name = f"{coin}_5years_15m.feather"
    
    df_coin = pd.read_feather(file_name)
    
    df_coin = df_coin.sort_values("OpenTime")
    
    df_coin = df_coin.set_index("OpenTime")
    
    dfs.append(df_coin)

print("Loaded all coins.")


Loaded all coins.


In [21]:
# Combine all coins
combined_df = pd.concat(dfs)

# Pivot to multi-coin matrix
matrix = combined_df.pivot_table(
    index="OpenTime",
    columns="Symbol",
    values=["Open", "High", "Low", "Close", "Volume"]
)

matrix = matrix.sort_index()

matrix.head()


Close                         High                     \
Symbol                BNBUSDT   BTCUSDT  ETHUSDT   BNBUSDT   BTCUSDT  ETHUSDT   
OpenTime                                                                        
2021-02-24 06:00:00  240.2982  49736.62  1618.05  248.3040  50719.78  1648.99   
2021-02-24 06:15:00  235.8546  49666.93  1612.79  242.5252  50147.00  1634.72   
2021-02-24 06:30:00  241.3975  50055.59  1627.75  241.8185  50076.92  1628.00   
2021-02-24 06:45:00  242.9589  50109.21  1637.60  244.8000  50427.00  1643.01   
2021-02-24 07:00:00  240.2269  49552.56  1614.58  245.0140  50181.33  1643.21   

                          Low                         Open                     \
Symbol                BNBUSDT   BTCUSDT  ETHUSDT   BNBUSDT   BTCUSDT  ETHUSDT   
OpenTime                                                                        
2021-02-24 06:00:00  239.7777  49593.46  1616.18  243.5559  50247.61  1627.42   
2021-02-24 06:15:00  235.5344  49456.00  1610.05  240.3751  49736.62  1617.69   
2021-02-24 06:30:00  235.8039  49573.42  1610.00  235.9794  49666.92  1612.40   
2021-02-24 06:45:00  240.5307  49927.57  1624.74  241.4877  50055.59  1627.31   
2021-02-24 07:00:00  240.2200  49548.47  1614.36  242.9551  50109.20  1637.58   

                         Volume                            
Symbol                  BNBUSDT      BTCUSDT      ETHUSDT  
OpenTime                                                   
2021-02-24 06:00:00  151180.501  1557.357006  13346.38967  
2021-02-24 06:15:00  182855.252  1309.230220  14505.60860  
2021-02-24 06:30:00   94903.977   926.348762  10067.88222  
2021-02-24 06:45:00  113713.304   873.182482  12905.22595  
2021-02-24 07:00:00   94152.152   779.167489  17812.04168

# VALIDATION

In [25]:
print("Matrix Shape:", matrix.shape)
print("Total Missing Values:", matrix.isnull().sum().sum())
print("Start Date:", matrix.index.min())
print("End Date:", matrix.index.max())


Matrix Shape: (175135, 15)
Total Missing Values: 0
Start Date: 2021-02-24 06:00:00
End Date: 2026-02-23 05:45:00


In [26]:
matrix.to_feather("crypto_matrix_5years_15m.feather")


In [27]:
from pathlib import Path

save_dir = Path("data/crypto_15min_feather")
save_dir.mkdir(parents=True, exist_ok=True)


In [28]:
start_date = matrix.index.min().strftime("%Y-%m-%d")
end_date   = matrix.index.max().strftime("%Y-%m-%d")


In [29]:
saved_files = []

for symbol in matrix.columns.levels[1]:

    coin_df = matrix.xs(symbol, level=1, axis=1).copy()
    coin_df = coin_df.reset_index()

    # Convert BTCUSDT → BTC-USDT
    formatted_symbol = symbol.replace("USDT", "-USDT")

    filename = (
        f"{formatted_symbol}-VANILLA-PERPETUAL"
        f"__15min__{start_date}__{end_date}.feather"
    )

    file_path = save_dir / filename

    coin_df.to_feather(file_path)

    saved_files.append(file_path)

saved_files


[WindowsPath('data/crypto_15min_feather/BNB-USDT-VANILLA-PERPETUAL__15min__2021-02-24__2026-02-23.feather'),
 WindowsPath('data/crypto_15min_feather/BTC-USDT-VANILLA-PERPETUAL__15min__2021-02-24__2026-02-23.feather'),
 WindowsPath('data/crypto_15min_feather/ETH-USDT-VANILLA-PERPETUAL__15min__2021-02-24__2026-02-23.feather')]

# 6 MONTH SPLIT SYSTEM

In [41]:
# ==========================================================
# COMPLETE 5-YEAR SAVE + 6-MONTH SPLIT SYSTEM
# ==========================================================

from pathlib import Path
import pandas as pd

#  CREATE FOLDERS

full_dir = Path("data/crypto_15min_feather")
split_dir = Path("data/crypto_15min_feather_6month_split")

full_dir.mkdir(parents=True, exist_ok=True)
split_dir.mkdir(parents=True, exist_ok=True)

interval = "15min"


#  GET FULL DATA DATE RANGE

start_full = matrix.index.min()
end_full   = matrix.index.max()

start_str_full = start_full.strftime("%Y%m%d")
end_str_full   = end_full.strftime("%Y%m%d")


# 3️ PROCESS EACH COIN

for symbol in matrix.columns.levels[1]:

    print("\n===================================================")
    print(f"Processing {symbol}_VANILLA_PERPETUAL_{interval}_{start_str_full}_{end_str_full}.feather")

    # Extract coin data
    coin_df = matrix.xs(symbol, level=1, axis=1).copy()
    coin_df = coin_df.reset_index()

    
    #  SAVE FULL 5-YEAR FILE
    

    full_filename = (
        f"{symbol}_VANILLA_PERPETUAL_{interval}_{start_str_full}_{end_str_full}.feather"
    )

    full_path = full_dir / full_filename
    coin_df.to_feather(full_path)

    print(f"Full File Saved → {full_filename} | Rows: {len(coin_df)}")

   
    #  SPLIT INTO 6-MONTH CHUNKS
   

    current_start = start_full

    while current_start < end_full:

        current_end = current_start + pd.DateOffset(months=6)

        chunk = coin_df[
            (coin_df["OpenTime"] >= current_start) &
            (coin_df["OpenTime"] < current_end)
        ]

        if chunk.empty:
            break

        chunk_start_str = current_start.strftime("%Y-%m-%d")
        chunk_end_str   = chunk["OpenTime"].max().strftime("%Y-%m-%d")

        chunk_filename = (
            f"{symbol}_VANILLA_PERPETUAL_{interval}_{start_str_full}_{end_str_full}"
            f"__{interval}__{chunk_start_str}__{chunk_end_str}.feather"
        )

        chunk_path = split_dir / chunk_filename
        chunk.to_feather(chunk_path)

        print(f"Saved: {chunk_filename} | Rows: {len(chunk)}")

        # Move to next 6-month window
        current_start = current_end

print("\n ALL COINS PROCESSED SUCCESSFULLY")


Processing BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223.feather
Full File Saved → BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223.feather | Rows: 175135
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2021-02-24__2021-08-24.feather | Rows: 17324
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2021-08-24__2022-02-24.feather | Rows: 17656
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2022-02-24__2022-08-24.feather | Rows: 17376
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2022-08-24__2023-02-24.feather | Rows: 17664
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2023-02-24__2023-08-24.feather | Rows: 17371
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2023-08-24__2024-02-24.feather | Rows: 17664
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2024-02-24__2024-08-24.feather | Rows: 17472
Saved: BNBUSDT_VANILLA_PERPETUAL_15min_20210224_20260223__15min__2024